# 01 Extraction

This notebook demonstrates the extraction of a balanced sample from the raw ~7.7M row US Accidents dataset.

## Objectives:
1. Load the raw dataset (in chunks to manage memory).
2. Analyze the actual distribution of the `Severity` column.
3. Extract a balanced sample of ~100,000 rows, ensuring sufficient representation across all severity levels and maintaining geographic diversity.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import extract_balanced_sample

### Raw Data Inspection
Let's look at the shape and columns of the raw dataset.

In [2]:
RAW_PATH = PROJECT_ROOT / 'data/raw/US_Accidents_March23.csv'

# Read just the first few rows to understand the structure
df_preview = pd.read_csv(RAW_PATH, nrows=5)
print(f"Columns: {len(df_preview.columns)}")
df_preview.head()

Columns: 46


,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,NaN,NaN,0.01,...,False,False,False,False,True,False,Night,Night,Day,Day
3,A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.205582,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Day,Day,Day
4,A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,NaN,NaN,0.01,...,False,False,False,False,True,False,Day,Day,Day,Day


### Balanced Extraction

The dataset is highly imbalanced. Severity 2 accounts for the vast majority of accidents, while Severity 1 and 4 are relatively rare. 
To build a robust analysis (and potentially models) without bias towards the majority class, we extract a balanced sample.

**Strategy:**
- Severity 1 target: 10,000
- Severity 2 target: 35,000
- Severity 3 target: 35,000
- Severity 4 target: 20,000
- Total target: ~100,000

We use the `extract_balanced_sample` function from our ETL pipeline, which uses stratified sampling by `State` within each severity level to maintain geographic diversity.

In [3]:
EXTRACTED_PATH = PROJECT_ROOT / 'data/processed/extracted_sample.csv'

# Note: Running this will take a few minutes as it processes the 3GB file.
# If the file already exists, we will just load it to save time.
if EXTRACTED_PATH.exists():
    print(f"Loading existing extracted sample from {EXTRACTED_PATH}")
    df_sampled = pd.read_csv(EXTRACTED_PATH)
else:
    print("Extracting sample from raw data...")
    df_sampled = extract_balanced_sample(RAW_PATH, EXTRACTED_PATH)

Loading existing extracted sample from /Users/mohitsingh/Desktop/DVA-capstone/Section-A_G12_DeliverIQ/data/processed/extracted_sample.csv


### Sample Verification

In [4]:
print(f"Extracted sample shape: {df_sampled.shape}")
print("\nSeverity Distribution:")
print(df_sampled['Severity'].value_counts().sort_index())

Extracted sample shape: (99998, 46)

Severity Distribution:
Severity
1    10000
2    35000
3    34999
4    19999
Name: count, dtype: int64


In [5]:
print(f"Number of unique states in sample: {df_sampled['State'].nunique()}")

Number of unique states in sample: 49
